In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from sklearn.model_selection import train_test_split
import re
import os

In [ ]:
def preprocess_text(text):
    """
    Normaliza el texto: minúsculas, reemplaza números por <NUM>, quita espacios extra.
    """
    if not isinstance(text, str): return ""
    text = text.lower()
    # Regex para encontrar números (enteros, decimales, comas)
    text = re.sub(r'\d[\d,.]*\d|\d', '<NUM>', text)
    # Reemplazar múltiples espacios
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['concepto_limpio'] = df['concepto'].apply(preprocess_text)

In [ ]:
TOKENIZER_FILE = "tokenizer_transacciones.json"

# Configuración del tokenizador BPE
tokenizer = Tokenizer(BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    special_tokens=["<unk>", "<pad>", "<NUM>"], 
    vocab_size=2000, 
    min_frequency=2
)

# Entrenamos con tus datos actuales
print("--- Entrenando Tokenizador ---")
tokenizer.train_from_iterator(df['concepto_limpio'].tolist(), trainer=trainer)
tokenizer.save(TOKENIZER_FILE)
#print("Tokenizador guardado.")

# Ajustamos el PAD_IDX para usarlo luego
PAD_IDX = tokenizer.token_to_id("<pad>")

In [ ]:
class TransactionDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=50):
        self.df = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Texto crudo
        text = self.df.iloc[idx]['concepto'] 
        # Etiqueta (En tu JSON ya viene como 0 o 1 en 'clasificacion')
        label = self.df.iloc[idx]['clasificacion']
        
        # 1. Preprocesar
        clean_text = preprocess_text(text)
        
        # 2. Tokenizar
        encoding = self.tokenizer.encode(clean_text)
        
        # Convertir a tensores
        return torch.tensor(encoding.ids), torch.tensor(label, dtype=torch.long)

def collate_batch(batch):
    label_list, text_list = [], []
    for (_text_ids, _label) in batch:
        label_list.append(_label)
        text_list.append(_text_ids)
    
    # Padding dinámico (rellena con PAD_IDX para que todos tengan el mismo largo)
    text_padded = pad_sequence(text_list, batch_first=True, padding_value=PAD_IDX)
    label_tensor = torch.tensor(label_list)
    
    return text_padded, label_tensor

# División Train/Test
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Crear Loaders
train_dataset = TransactionDataset(train_df, tokenizer)
test_dataset = TransactionDataset(test_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collate_batch)

In [ ]:
import numpy as np
import torch

class EarlyStopping:
    def __init__(self, patience=3, verbose=False, delta=0, path='mejor_modelo.pt'):
        """
        Args:
            patience (int): Cuántas épocas esperar después de que la loss deje de mejorar.
            verbose (bool): Si es True, imprime mensajes cada vez que guarda.
            delta (float): Cambio mínimo para considerar una mejora.
            path (str): Dónde guardar el archivo del modelo (checkpoint).
        """
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.Inf
        self.delta = delta
        self.path = path

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        '''Guarda el modelo cuando la loss de validación disminuye.'''
        if self.verbose:
            print(f'Validación Loss mejoró ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Guardando modelo...')
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embedding_dim, 
                            hidden_dim, 
                            num_layers=2, 
                            bidirectional=True, 
                            batch_first=True, 
                            dropout=0.3)
        # Hidden dim * 2 porque es Bidireccional
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(0.5)
        
    def forward(self, text):
        # text = [batch size, sent len]
        embedded = self.dropout(self.embedding(text))
        
        # outputs, (hidden, cell)
        # Usamos packed_sequence si quisieramos optimizar, pero aquí seguimos tu lógica simple
        output, (hidden, cell) = self.lstm(embedded)
        
        # Concatenamos el último estado oculto forward y backward
        # hidden = [num layers * num directions, batch size, hid dim]
        hidden_final = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        
        return self.fc(hidden_final)

# Hiperparámetros
VOCAB_SIZE = tokenizer.get_vocab_size()
EMBEDDING_DIM = 100
HIDDEN_DIM = 128
OUTPUT_DIM = 2 # Porque tienes clasificacion 0 y 1

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = BiLSTMClassifier(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM, PAD_IDX)
model = model.to(device)

In [ ]:
# Configuración
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss().to(device)

# Instanciamos el Early Stopping
# patience=3: Si en 3 épocas no mejora, se detiene.
early_stopping = EarlyStopping(patience=3, verbose=True, path='modelo_bilstm_final.pt')

print(f"\n--- Iniciando Entrenamiento Inteligente en {device} ---")

num_epochs = 20 # Podemos poner muchas, total el Early Stopping lo parará

for epoch in range(num_epochs):
    # --- FASE 1: ENTRENAMIENTO ---
    model.train()
    train_loss = 0
    
    for text, labels in train_loader:
        text, labels = text.to(device), labels.to(device)
        
        optimizer.zero_grad()
        predictions = model(text)
        loss = criterion(predictions, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    avg_train_loss = train_loss / len(train_loader)

    # --- FASE 2: VALIDACIÓN (Sin actualizar pesos) ---
    model.eval() 
    val_loss = 0
    
    with torch.no_grad(): # Desactiva el cálculo de gradientes para ahorrar memoria
        for text, labels in test_loader:
            text, labels = text.to(device), labels.to(device)
            predictions = model(text)
            loss = criterion(predictions, labels)
            val_loss += loss.item()
            
    avg_val_loss = val_loss / len(test_loader)
    
    print(f'Epoch {epoch+1:02} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}')
    
    # --- FASE 3: EARLY STOPPING ---
    # Le pasamos la pérdida de validación actual
    early_stopping(avg_val_loss, model)
    
    if early_stopping.early_stop:
        print(" Early stopping activado. Se detiene el entrenamiento.")
        break


# Importante: Al final, cargamos el mejor modelo guardado, no el de la última época
model.load_state_dict(torch.load('modelo_bilstm_final.pt'))


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
import torch.nn.functional as F

def plot_roc_curve(model, data_loader, device):
    model.eval() # Modo evaluación
    
    all_probs = []
    all_labels = []
    
    print("Generando predicciones para ROC...")
    
    with torch.no_grad():
        for text, labels in data_loader:
            text = text.to(device)
            labels = labels.to(device)
            
            # 1. Obtener logits del modelo
            logits = model(text)
            
            # 2. Aplicar Softmax para tener probabilidades (0 a 1)
            # dim=1 es la dimensión de las clases. 
            probs = F.softmax(logits, dim=1)
            
            # 3. Nos quedamos SOLO con la probabilidad de la Clase 1 
            # (Asumiendo 0=Retiro, 1=Deposito, esto grafica qué tan bueno es detectando Depósitos)
            probs_clase_1 = probs[:, 1]
            
            # 4. Guardar en CPU para sklearn
            all_probs.extend(probs_clase_1.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # --- CÁLCULO DE MÉTRICAS ---
    # fpr = Tasa de Falsos Positivos
    # tpr = Tasa de Verdaderos Positivos (Sensibilidad)
    fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
    roc_auc = auc(fpr, tpr)

    # --- GRAFICAR ---
    plt.figure(figsize=(10, 8))
    
    # Curva del modelo
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Curva ROC (AUC = {roc_auc:.4f})')
    
    # Línea de azar (diagonal punteada) - Un modelo que adivina estaría aquí
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Tasa de Falsos Positivos (1 - Especificidad)')
    plt.ylabel('Tasa de Verdaderos Positivos (Sensibilidad)')
    plt.title('Receiver Operating Characteristic (ROC) - Test Set')
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.show()

# --- EJECUTAR ---
# Asegúrate de que 'model' ya tenga cargado el mejor estado del Early Stopping
plot_roc_curve(model, train_loader, device)

In [ ]:

class MotorInferencia:
    def __init__(self, model_path, tokenizer_path, device='cpu'):
        self.device = device
        
        # A. Cargar Tokenizador
        print(f"Cargando tokenizador desde {tokenizer_path}...")
        self.tokenizer = Tokenizer.from_file(tokenizer_path)
        self.pad_idx = self.tokenizer.token_to_id("<pad>")
        
        # B. Cargar Modelo
        print(f"Cargando modelo desde {model_path}...")
        # Parámetros (Deben coincidir con los de entrenamiento)
        VOCAB_SIZE = self.tokenizer.get_vocab_size()
        EMBEDDING_DIM = 100
        HIDDEN_DIM = 128
        OUTPUT_DIM = 2
        
        self.model = BiLSTMClassifier(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM, self.pad_idx)
        
        # Cargar los pesos guardados (state_dict)
        # map_location asegura que cargue en CPU si no hay GPU disponible
        self.model.load_state_dict(torch.load(model_path, map_location=device))
        self.model.to(device)
        self.model.eval() # IMPORTANTE: Poner en modo evaluación
        
    def _preprocess(self, text):
        """Misma limpieza que en el entrenamiento"""
        if not isinstance(text, str): return ""
        text = text.lower()
        text = re.sub(r'\d[\d,.]*\d|\d', '<NUM>', text) # Reemplazar números
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def predecir(self, texto_bruto):
        # 1. Limpieza
        texto_limpio = self._preprocess(texto_bruto)
        
        # 2. Tokenización
        # encode devuelve un objeto Encoding, queremos los ids
        ids = self.tokenizer.encode(texto_limpio).ids
        
        # 3. Convertir a Tensor y agregar dimensión de Batch
        # De [longitud_frase] pasamos a [1, longitud_frase]
        tensor_text = torch.tensor([ids], dtype=torch.long).to(self.device)
        
        # 4. Inferencia
        with torch.no_grad():
            logits = self.model(tensor_text)
            probabilities = F.softmax(logits, dim=1)
            
            # Obtener la clase ganadora y su probabilidad
            confianza, clase_idx = torch.max(probabilities, dim=1)
            
        return {
            "texto_original": texto_bruto,
            "texto_procesado": texto_limpio,
            "clase_predicha": clase_idx.item(), # 0 o 1
            "confianza": confianza.item() # Valor entre 0 y 1
        }

# --- EJEMPLO DE USO ---

# Rutas de tus archivos guardados
RUTA_MODELO = r'\modelo_bilstm_final.pt'
RUTA_TOKENIZER = r'\tokenizer_transacciones.json'

# Inicializamos el motor (solo se hace una vez)
engine = MotorInferencia(RUTA_MODELO, RUTA_TOKENIZER)


In [ ]:

# Lista de pruebas
nuevas_transacciones = [
"Amanda Cristina Vazquez Zurita PAGO CUENTA DE TERCERO BNET 0479691103 nomina amanda" , 
"PAGO RECIBIDO DE KUSPIT POR ORDEN DE ARRENDADORA PANORAMA SA DE CV CTA.ORDENANTE 653180003810089857 REF.0000001 SUC 0 CAJA 0 AUT 0 HORA 0:0 PGO RASTREO: UNALANAPAY0052663829", 
"SPEI RECIBIDOAZTECA 3846270terapia Dario Rivera 00127540013554848931 2504140708281212211 GARCIA RODRIGUEZ CLAUDIA ISABEL"

]

print("\n--- RESULTADOS DE INFERENCIA ---")
for tx in nuevas_transacciones:
    resultado = engine.predecir(tx)
    
    # Mapeo visual (Ajusta según qué es 0 y qué es 1 en tus datos)
    etiqueta = "Retiro" if resultado['clase_predicha'] == 1 else "Deposito"
    
    print(f"Tx: {resultado['texto_original']}")
    print(f"   -> Predicción: {etiqueta}")
    print(f"   -> Confianza:  {resultado['confianza']:.2%}\n")

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.nn.functional as F
import torch
import pandas as pd
import numpy as np

# 1. Definimos un Dataset específico para Inferencia
#    (A diferencia del de entrenamiento, este NO necesita devolver etiquetas, solo texto)
class InferenceDataset(Dataset):
    def __init__(self, texts, tokenizer):
        self.texts = texts
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        
        # A. Preprocesamiento (Limpieza igual al entrenamiento)
        clean_text = preprocess_text(text) 
        
        # B. Tokenización
        ids = self.tokenizer.encode(clean_text).ids
        
        return torch.tensor(ids, dtype=torch.long)

# 2. Función Collate para hacer Padding en los lotes
def collate_inference(batch):
    # batch es una lista de tensores [tensor(ids), tensor(ids), ...]
    # pad_idx lo sacamos del tokenizador
    pad_idx = tokenizer.token_to_id("<pad>")
    
    # Rellenamos para que todos tengan el mismo largo en este lote
    text_padded = pad_sequence(batch, batch_first=True, padding_value=pad_idx)
    return text_padded

# 3. Función Principal para procesar el DataFrame
def predecir_dataframe(df, col_texto, model, tokenizer, batch_size=32, device='cpu'):
    
    # Preparar datos
    textos = df[col_texto].astype(str).tolist() # Aseguramos que sea string
    dataset = InferenceDataset(textos, tokenizer)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_inference)
    
    model.eval() # Modo evaluación
    model.to(device)
    
    todas_predicciones = []
    todas_confianzas = []
    
    with torch.no_grad():
        for batch_text in loader:
            batch_text = batch_text.to(device)
            
            # Forward
            logits = model(batch_text)
            
            # Probabilidades
            probs = F.softmax(logits, dim=1)
            
            # Obtener clase ganadora y su probabilidad
            max_probs, clases = torch.max(probs, dim=1)
            
            # Guardar en listas (pasando a CPU)
            todas_predicciones.extend(clases.cpu().numpy())
            todas_confianzas.extend(max_probs.cpu().numpy())
            
    return todas_predicciones, todas_confianzas

# =======================================================
# EJECUCIÓN
# =======================================================

# Definir dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Supongamos que tu dataframe se llama 'df' y la columna de texto es 'concepto'
# batch_size=64 es una buena velocidad. Si te da error de memoria, bájalo a 32 o 16.
preds, confs = predecir_dataframe(df, 'concepto', model, tokenizer, batch_size=64, device=device)

# 4. Agregamos los resultados al DataFrame original
df['prediccion_clase'] = preds
df['probabilidad'] = confs

# Opcional: Mapear el 0 y 1 a texto si quieres
mapa_clases = {1: 'RETIRO', 0: 'DEPOSITO'} # Ajusta esto según tu entrenamiento real
df['prediccion_texto'] = df['prediccion_clase'].map(mapa_clases)





In [ ]:
df_regex = df[['concepto', 'prediccion_texto', 'probabilidad',"clasificacion"]].loc[df.clasificacion== 0].copy()

In [ ]:
import pandas as pd
import numpy as np
import unicodedata

# 1. FUNCIÓN DE LIMPIEZA (Para ignorar mayúsculas y acentos)
def normalizar_texto(texto):
    if not isinstance(texto, str):
        return ""
    # Pasa a minúsculas
    texto = texto.lower()
    # Quita acentos (á -> a, é -> e, etc.)
    texto = ''.join((c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn'))
    return texto

reglas_exclusion = {
    'INVERSION_VENCIDA': r"inversion|plazo fijo|pagare|rendimiento|cetes|bonddia",
    'DEVOLUCION_REVERSO': r"devolucion|reverso|devuelto|reembolso|cancelacion|rechazado",
    'PRESTAMO_CREDITO': r"prestamo|credito|linea revolvente|disposicion|abono a capital|financiamiento|tdc",
    'TRASPASO_PROPIO': r"traspaso entre cuentas|misma cuenta|cuenta propia|traspaso a terceros", # Ajusta según tu necesidad
    'FUERA_FLUJO': r"reingreso|error contable|correccion"
}



# A. Normalizamos la columna para facilitar la búsqueda
df_regex['concepto_norm'] = df_regex['concepto'].apply(normalizar_texto)

# B. Inicializamos columnas por defecto
df_regex['decision_final'] = 'INCLUIR'
df_regex['razon_exclusion'] = ''

# C. Aplicamos las reglas
print("Aplicando reglas regex...")

for motivo, patron in reglas_exclusion.items():
    # Buscamos filas que contengan el patrón (case=False ya no es necesario por la normalización, pero no estorba)
    filtro = df_regex['concepto_norm'].str.contains(patron, case=False, regex=True, na=False)
    
    # Marcamos esas filas como EXCLUIR y guardamos la razón
    # Solo sobrescribimos si aún no ha sido marcado (para quedarnos con la primera razón encontrada)
    mask_pendientes = (df_regex['decision_final'] == 'INCLUIR') & filtro
    
    df_regex.loc[mask_pendientes, 'decision_final'] = 'EXCLUIR'
    df_regex.loc[mask_pendientes, 'razon_exclusion'] = motivo


# Ver los excluidos
excluidos = df_regex[df_regex['decision_final'] == 'EXCLUIR']
excluidos.sample(10)